# bagging和随机森林

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine              # 葡萄酒分类数据集（178样本，13特征，3类）
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score           # 准确率评估指标
from sklearn.tree import DecisionTreeClassifier      # 决策树分类器（作为基学习器）
import matplotlib.pyplot as plt

In [ ]:
# 加载葡萄酒数据集：178个样本，13个化学特征，3个类别
wine = load_wine()
print(f"所有特征：{wine.feature_names}")
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target)
# 划分训练集(80%)和测试集(20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1)

In [ ]:
# 构建单棵决策树作为基学习器（弱分类器）
# max_depth=1：限制树深度为1（即决策树桩，只有一个划分节点），作为弱学习器
# criterion='gini'：使用基尼系数作为划分准则，Gini(D) = 1 - sum(p_k^2)
base_model = DecisionTreeClassifier(max_depth=1, criterion='gini',random_state=1).fit(X_train, y_train)
y_pred = base_model.predict(X_test)  # 在测试集上预测
print(f"决策树的准确率：{accuracy_score(y_test,y_pred):.3f}")
# 单棵决策树桩的准确率较低（约0.694），说明弱学习器需要通过集成来提升性能

## bagging

In [ ]:
# Bagging（Bootstrap Aggregating）集成方法
# 核心思想：通过有放回抽样（Bootstrap）生成多个子训练集，分别训练基学习器，再投票/平均
# 约63.2%的原始样本会出现在每个Bootstrap采样集中（其余约36.8%为袋外样本OOB）
# Bagging 通过降低模型方差来提升泛化性能，特别适合高方差的基学习器（如决策树）
from sklearn.ensemble import BaggingClassifier
model = BaggingClassifier(base_estimator=base_model,  # 基学习器：前面定义的决策树桩
                            n_estimators=50,            # 集成50个基学习器
                            random_state=1)
model.fit(X_train, y_train)  # 训练
y_pred = model.predict(X_test)  # 预测：对所有基学习器的预测结果进行多数投票
print(f"BaggingClassifier的准确率：{accuracy_score(y_test,y_pred):.3f}")
# 集成后准确率从0.694提升到0.917，体现了集成学习的威力

## 测试估计器个数的影响

In [ ]:
# 探索基学习器数量（n_estimators）对 Bagging 性能的影响
# 理论上，基学习器越多，集成效果越好，但边际收益递减，且计算成本增加
x = list(range(2, 102, 2))  # 估计器个数从2到100，步长为2
y = []

for i in x:
    model = BaggingClassifier(base_estimator=base_model,
                              n_estimators=i,
                              random_state=1)
    model.fit(X_train, y_train)
    model_test_sc = accuracy_score(y_test, model.predict(X_test))
    y.append(model_test_sc)

# 绘制学习曲线：观察准确率随基学习器数量的变化趋势
plt.style.use('ggplot')
plt.title("Effect of n_estimators", pad=20)
plt.xlabel("Number of base estimators")  # 基学习器数量
plt.ylabel("Test accuracy of BaggingClassifier")  # 测试集准确率
plt.plot(x, y)
plt.show()

## 随机森林

In [ ]:
# 随机森林（Random Forest）
# 本质是 Bagging + 随机特征选择：在 Bagging 基础上，每次划分节点时
# 不是从全部 d 个特征中选最优，而是随机选择 k 个特征的子集（推荐 k = log2(d)）
# 这进一步增加了基学习器之间的差异性，通常比纯 Bagging 效果更好
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
                            n_estimators=50,  # 使用50棵决策树
                            random_state=1)
model.fit(X_train, y_train)  # 训练
y_pred = model.predict(X_test)  # 预测
print(f"RandomForestClassifier的准确率：{accuracy_score(y_test,y_pred):.3f}")
# 随机森林准确率（0.972）高于 Bagging（0.917），体现了随机特征选择的优势

In [ ]:
# 探索基学习器数量对随机森林性能的影响
# 随机森林同样受益于更多的基学习器，且收敛速度通常比 Bagging 更快
x = list(range(2, 102, 2))
y = []

for i in x:
    model = RandomForestClassifier(
                              n_estimators=i,
                              random_state=1)
    model.fit(X_train, y_train)
    model_test_sc = accuracy_score(y_test, model.predict(X_test))
    y.append(model_test_sc)

plt.style.use('ggplot')
plt.title("Effect of n_estimators", pad=20)
plt.xlabel("Number of base estimators")
plt.ylabel("Test accuracy of RandomForestClassifier")
plt.plot(x, y)
plt.show()